In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Baseline logistic 잡고 어떻게 성능을 개선할지 (logistic reg 자체 성능 개선)
- feature selection
- threshold tuning
- 성능 개선

In [ ]:
# 1. import numpy, pandas, matplotlib, seaborn
import numpy as np
import pandas as pd


# 2. matplotlib inline 설정
%matplotlib inline

In [ ]:
data_processed_parquet_path = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.parquet"
df = pd.read_parquet(data_processed_parquet_path)

In [ ]:

from sklearn.model_selection import train_test_split #data split to train and test
from sklearn.pipeline import Pipeline #여러 단계 묶기
from sklearn.impute import SimpleImputer # 결측치 채우는 도구
from sklearn.preprocessing import StandardScaler #Feature Scaling mean:0, var:1 값 범위 맞추기
from sklearn.linear_model import LogisticRegression #baseline model

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


In [ ]:
df.shape
df.head()

# baseline logistic regression을 먼저 구축해 기준 성능을 확인
# 이후 결측치 비율이 높은 센서와 분산이 거의 없는 센서를 제거
# 추가로 SelectKBest를 사용해 informative feature만 선택
# 그 결과 baseline 대비 precision / recall / F1 변화를 확인
# 이는 제조 센서 데이터에서 feature 정제가 모델 성능 개선에 유의미

,sample_id,timestamp,sensor_001,sensor_002,sensor_003,sensor_004,sensor_005,sensor_006,sensor_007,sensor_008,...,sensor_582,sensor_583,sensor_584,sensor_585,sensor_586,sensor_587,sensor_588,sensor_589,sensor_590,label
0,sample_0001,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,sample_0002,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,sample_0003,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,sample_0004,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,sample_0005,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [ ]:
target_col = "label"
drop_cols = ["sample_id", "timestamp"]

# 필요 없는 컬럼 제거
# if drop_cols exist, put it in the list
drop_cols_exist = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=drop_cols_exist, errors="ignore")

# 타겟 존재 확인
if target_col not in df.columns:
    raise ValueError(f"'{target_col}' column not found in dataframe.")

# X, Y 분리
X = df.drop(columns=[target_col])
y = df[target_col].copy()


print("Target distribution:")
print(y.value_counts(dropna=False))
print(df.shape)


Target distribution:
label
-1    1463
 1     104
Name: count, dtype: int64
(1567, 591)


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
print(X_train.shape)
print(y_train.shape)

(1253, 590)
(1253,)


In [ ]:
#Chekcing null values ratio per sensors
missing_ratio = X_train.isnull().mean().sort_values(ascending=False)
missing_ratio.head(20)

,0
sensor_293,0.906624
sensor_294,0.906624
sensor_159,0.906624
sensor_158,0.906624
sensor_221,0.856345
sensor_086,0.856345
sensor_359,0.856345
sensor_493,0.856345
sensor_385,0.663208
sensor_383,0.663208


In [ ]:
#remove sensors that have missing ratio over 0.5
high_missing_cols = missing_ratio[missing_ratio > 0.5].index.tolist()

print("제거할 컬럼 수:", len(high_missing_cols))

X_train_1 = X_train.drop(columns=high_missing_cols)
X_test_1 = X_test.drop(columns=high_missing_cols)

print(X_train_1.shape, X_test_1.shape)

제거할 컬럼 수: 24
(1253, 566) (314, 566)


In [ ]:
#정보량 거의 없는 sensors 제거, 분산 거의 없는 feature 제거
imputer_for_variance = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer_for_variance.fit_transform(X_train_1),
    columns=X_train_1.columns,
    index=X_train_1.index
)

selector_var = VarianceThreshold(threshold=0.0)
selector_var.fit(X_train_imputed)

selected_var_cols = X_train_imputed.columns[selector_var.get_support()].tolist()

X_train_2 = X_train_1[selected_var_cols].copy()
X_test_2 = X_test_1[selected_var_cols].copy()

print("분산 필터 후 feature 수:", len(selected_var_cols))

분산 필터 후 feature 수: 450


In [ ]:
#Baseline improved logistic regression 돌ㄹ리기
#일단 결측치 많은 컬럼 제거 + low variance 제거까지만 반영한 logistic를 돌려본다.
pipe_logistic_v1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

pipe_logistic_v1.fit(X_train_2, y_train)
pred_v1 = pipe_logistic_v1.predict(X_test_2)

In [ ]:

print("Accuracy:", accuracy_score(y_test, pred_v1))
print("Precision:", precision_score(y_test, pred_v1, zero_division=0))
print("Recall:", recall_score(y_test, pred_v1, zero_division=0))
print("F1:", f1_score(y_test, pred_v1, zero_division=0))
print()
print(classification_report(y_test, pred_v1, zero_division=0))
print(confusion_matrix(y_test, pred_v1))

Accuracy: 0.8407643312101911
Precision: 0.10810810810810811
Recall: 0.19047619047619047
F1: 0.13793103448275862

              precision    recall  f1-score   support

          -1       0.94      0.89      0.91       293
           1       0.11      0.19      0.14        21

    accuracy                           0.84       314
   macro avg       0.52      0.54      0.53       314
weighted avg       0.88      0.84      0.86       314

[[260  33]
 [ 17   4]]


In [ ]:
#8. feature selection 추가 실험
#상위 feature만 뽑아보기
# k=50, k=100, k=150
imputer_fs = SimpleImputer(strategy="median")

X_train_fs_input = pd.DataFrame(
    imputer_fs.fit_transform(X_train_2),
    columns=X_train_2.columns,
    index=X_train_2.index
)

X_test_fs_input = pd.DataFrame(
    imputer_fs.transform(X_test_2),
    columns=X_test_2.columns,
    index=X_test_2.index
)

selector_k = SelectKBest(score_func=f_classif, k=50)
selector_k.fit(X_train_fs_input, y_train)

selected_k_cols = X_train_fs_input.columns[selector_k.get_support()].tolist()

X_train_3 = X_train_2[selected_k_cols].copy()
X_test_3 = X_test_2[selected_k_cols].copy()

print("선택된 feature 수:", len(selected_k_cols))
print(selected_k_cols[:10])

선택된 feature 수: 50
['sensor_015', 'sensor_022', 'sensor_023', 'sensor_029', 'sensor_060', 'sensor_064', 'sensor_065', 'sensor_066', 'sensor_071', 'sensor_096']


In [ ]:
#9. selected feature로 logistic 다시 학습
pipe_logistic_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

pipe_logistic_v2.fit(X_train_3, y_train)
pred_v2 = pipe_logistic_v2.predict(X_test_3)

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred_v2))
print("Precision:", precision_score(y_test, pred_v2, zero_division=0))
print("Recall:", recall_score(y_test, pred_v2, zero_division=0))
print("F1:", f1_score(y_test, pred_v2, zero_division=0))
print()
print(classification_report(y_test, pred_v2, zero_division=0))
print(confusion_matrix(y_test, pred_v2))

Accuracy: 0.7452229299363057
Precision: 0.12658227848101267
Recall: 0.47619047619047616
F1: 0.2

              precision    recall  f1-score   support

          -1       0.95      0.76      0.85       293
           1       0.13      0.48      0.20        21

    accuracy                           0.75       314
   macro avg       0.54      0.62      0.52       314
weighted avg       0.90      0.75      0.81       314

[[224  69]
 [ 11  10]]


In [ ]:
results = pd.DataFrame([
    {
        "experiment": "baseline_logistic",
        "feature_setting": "raw sensor only",
        "precision": None,   # baseline 값 넣기
        "recall": None,
        "f1": None
    },
    {
        "experiment": "improved_logistic_v1",
        "feature_setting": "drop high-missing + variance filter",
        "precision": precision_score(y_test, pred_v1, zero_division=0),
        "recall": recall_score(y_test, pred_v1, zero_division=0),
        "f1": f1_score(y_test, pred_v1, zero_division=0)
    },
    {
        "experiment": "improved_logistic_v2",
        "feature_setting": "v1 + SelectKBest",
        "precision": precision_score(y_test, pred_v2, zero_division=0),
        "recall": recall_score(y_test, pred_v2, zero_division=0),
        "f1": f1_score(y_test, pred_v2, zero_division=0)
    }
])

results

,experiment,feature_setting,precision,recall,f1
0,baseline_logistic,raw sensor only,NaN,NaN,NaN
1,improved_logistic_v1,drop high-missing + variance filter,0.108108,0.190476,0.137931
2,improved_logistic_v2,v1 + SelectKBest,0.126582,0.476190,0.200000


In [ ]:
feature_scores = pd.DataFrame({
    "feature": X_train_fs_input.columns,
    "score": selector_k.scores_
}).sort_values(by="score", ascending=False)

feature_scores.head(20)

,feature,score
95,sensor_104,38.345328
392,sensor_511,26.408693
54,sensor_060,20.145968
281,sensor_349,15.719980
118,sensor_130,14.664044
335,sensor_432,14.090072
59,sensor_065,13.675521
19,sensor_022,12.463758
114,sensor_126,11.306550
334,sensor_431,10.875200


In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/Semiconductor-AnomalyDetection"
processed_path = os.path.join(PROJECT_ROOT, "data", "processed")

print(os.listdir(processed_path))

#high missing removed, variance filtered, SelectKBest applied -> final trained feature
X_train_3.to_parquet(os.path.join(processed_path, "X_train_selected.parquet"))
X_test_3.to_parquet(os.path.join(processed_path, "X_test_selected.parquet"))

y_train.to_frame(name="label").to_parquet(os.path.join(processed_path, "y_train.parquet"))
y_test.to_frame(name="label").to_parquet(os.path.join(processed_path, "y_test.parquet"))


print(os.listdir(processed_path))


['.gitkeep', 'master_df.parquet', 'master_df.csv']
['.gitkeep', 'master_df.parquet', 'master_df.csv', 'X_train_selected.parquet', 'X_test_selected.parquet', 'y_train.parquet', 'y_test.parquet']


In [ ]:
!ls

drive  sample_data
